# Dataset Precomputation and Loading for Titans (Local GPU)

Локальная версия ноутбука, адаптированная для запуска на NVIDIA RTX 2060 SUPER (8 ГБ VRAM).

Этот ноутбук предназначен для фильтрации датасета TriviaQA, сохранения его в формате `ArrayRecord` и последующей быстрой загрузки для обучения модели Titans.

In [1]:
import os

# JAX: не захватывать всю VRAM сразу, выделять динамически
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Альтернатива: фиксированная доля VRAM (раскомментировать если preallocate=true)
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"

## Установка зависимостей

Клонируем и устанавливаем необходимые пакеты (запустить один раз).

In [2]:
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron

Cloning into 'kauldron'...
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [18 lines of output]
      Traceback (most recent call last):
        File "C:\Users\LiveComp\Titans\venv_312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
        File "C:\Users\LiveComp\Titans\venv_312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\LiveComp\Titans\venv_312\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\LiveComp\AppData\Local\Temp\pip-build-env-zqqljq0y\overlay\Lib\site-packages\setupto

In [ ]:
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

In [ ]:
import os
# os._exit(0)

In [ ]:
from gemma import gm
import jax
import jax.numpy as jnp

original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)


from gemma.gm.nn import _gemma

model = _gemma.Gemma3_1B(
    dtype=jnp.float16,  # float16 вместо bfloat16: аппаратная поддержка на Turing (RTX 20xx)
    # return_last_only=False,
    tokens="batch.tokens",
)

In [ ]:
import os
import grain.python as grain
from kauldron import kd
from etils import epath
import array_record.python.array_record_module as array_record
import tqdm
import pickle

# Константы
DATA_DIR = epath.Path(os.path.expanduser("~/Titans/data/trivia_qa_filtered"))  # Локальный путь
MAX_LENGTH = 2048 # длина последовательность в токенах
max_new_tokens = 64
BATCH_SIZE = 16 # Для сохранения не критично, но используется в оригинальном коде
MAX_CONTEXT_CHARS = (MAX_LENGTH - max_new_tokens) * 4
split_name = "validation"

DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Оригинальные компоненты и фильтрация

In [ ]:
def format_triviaqa_prompt(record):
    """Formats a TriviaQA record into a prompt for Gemma."""
    question = record.get('question', b'').decode('utf-8') if isinstance(record.get('question'), bytes) else record.get('question', '')

    # Extract context from search results if available
    contexts = record.get('search_results', {}).get('search_context', [])
    context_str = ""
    if contexts:
        ctx = contexts[0]
        context_str = ctx.decode('utf-8') if isinstance(ctx, bytes) else str(ctx)

    prompt = f"Context: {context_str}\n\nQuestion: {question}\n\nAnswer:"
    return prompt

In [ ]:
class FilterShortContext(grain.FilterTransform):
    def filter(self, x):
        ctxs = x.get('search_results', {}).get('search_context', [])
        if not ctxs:
            return False
        return len(ctxs[0]) <= MAX_CONTEXT_CHARS

def get_source_dataset(split_name):
    """Возвращает исходный TFDS датасет для фильтрации."""
    return kd.data.py.Tfds(
        name="trivia_qa/rc",
        split=split_name, ##splits: 'test' | 'train' | 'validation'
        shuffle=False, # Для сохранения порядок не важен
        num_epochs=1,  # Проходим один раз
        batch_size=None, # Читаем по одной записи
    )

## 2. Сохранение в ArrayRecord

Этот этап нужно запустить один раз.

In [ ]:
# split_name = 'test'
def precompute_and_save(split_name):
    ds = get_source_dataset(split_name)
    filter_transform = FilterShortContext()

    output_path = DATA_DIR / f"{split_name}.array_record"
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    print(f"Начинаю фильтрацию и сохранение в {output_path}...")

    count = 0
    # Итерируемся по сырым данным из TFDS
    for record in tqdm.tqdm(ds):
        if filter_transform.filter(record):
            # Сохраняем только нужные поля для экономии места
            serialized = pickle.dumps(record)
            writer.write(serialized)
            count += 1

    writer.close()
    print(f"Готово! Сохранено {count} валидных записей для {split_name}.")
# Раскомментируйте для запуска, если на гугл-диске еще нет отфильтрованных файлов .array_record с короткими контекстами
# for split_name in ['train', 'validation','test']:
#     precompute_and_save(split_name) # Раскомментируйте для запуска

## 5. Офлайн-генерация ответов Gemma

В этом разделе мы пропускаем отфильтрованные записи `trivia_qa/rc` через оригинальную модель `Gemma3_1B_IT` для генерации её собственных ответов (`dst`).

In [ ]:
input_file = f"{split_name}.array_record"
output_file = f"{split_name}_gemma_generated_full.array_record"
input_path = os.path.join(str(DATA_DIR), input_file)
output_path = os.path.join(str(DATA_DIR), output_file)
reader = array_record.ArrayRecordReader(str(input_path))
writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

In [ ]:
print(f"прочитано в {split_name} {reader.num_records()} записей")

In [ ]:
import array_record.python.array_record_module as array_record
import pickle
import tqdm
import kauldron as kd
from gemma import gm
import os
import jax
import threading
import queue
from concurrent.futures import ThreadPoolExecutor

NUM_WORKERS = min(os.cpu_count() or 4, 8)
NUM_PREFETCH = 2  # сколько батчей готовить заранее в очереди

def generate_and_save_batched_dataset(
    model,
    params,
    tokenizer,
    split_name,
    batch_size=4,
    max_new_tokens=64,
    limit=None
):
    input_path = os.path.join(str(DATA_DIR), f"{split_name}.array_record")
    output_path = os.path.join(str(DATA_DIR), f"{split_name}_gemma_generated.array_record")

    print(f"Обработка {input_path}...")
    print(f"Пул потоков: {NUM_WORKERS} воркеров, prefetch: {NUM_PREFETCH} батчей")

    sampler = gm.text.Sampler(model=model, params=params, tokenizer=tokenizer)
    reader = array_record.ArrayRecordReader(str(input_path))
    num_records = reader.num_records() if limit is None else min(limit, reader.num_records())
    writer = array_record.ArrayRecordWriter(str(output_path), options="group_size:1")

    max_prompt_tokens = MAX_LENGTH - max_new_tokens  # 2048 - 64 = 1984

    # Thread-local токенизатор (SentencePiece может быть не потокобезопасным)
    _tls = threading.local()

    def process_record(raw_bytes):
        """Десериализация + форматирование + усечение одной записи (в потоке)."""
        if not hasattr(_tls, "tok"):
            _tls.tok = gm.text.Gemma3Tokenizer()
        record = pickle.loads(raw_bytes)
        prompt = format_triviaqa_prompt(record)
        tokens = _tls.tok.encode(prompt)
        if len(tokens) > max_prompt_tokens:
            tokens = tokens[-max_prompt_tokens:]  # хвост промпта важнее
        return record, _tls.tok.decode(tokens)

    # --- Prefetch pipeline: фоновый поток читает и обрабатывает батчи ---
    batch_queue = queue.Queue(maxsize=NUM_PREFETCH)
    stop_event = threading.Event()

    def batch_producer():
        """Читает записи из reader и обрабатывает их параллельно в пуле потоков."""
        pool = ThreadPoolExecutor(max_workers=NUM_WORKERS)
        try:
            for i in range(0, num_records, batch_size):
                if stop_event.is_set():
                    break
                count = min(batch_size, num_records - i)
                raw_list = [reader.read() for _ in range(count)]
                # Параллельная обработка записей в батче
                results = list(pool.map(process_record, raw_list))
                records = [r[0] for r in results]
                prompts = [r[1] for r in results]
                batch_queue.put((records, prompts))
        except Exception as e:
            batch_queue.put(e)
        finally:
            pool.shutdown(wait=True)
        batch_queue.put(None)  # сигнал завершения

    producer = threading.Thread(target=batch_producer, daemon=True)
    producer.start()

    # --- Основной цикл: GPU инференс + запись ---
    written = 0
    interrupted = False
    try:
        batch_idx = 0
        with tqdm.tqdm(total=num_records) as pbar:
            while True:
                item = batch_queue.get()
                if item is None:
                    break
                if isinstance(item, Exception):
                    raise item

                records, prompts = item

                if batch_idx % 3 == 0:
                    jax.clear_caches()

                if prompts:
                    responses = sampler.sample(prompts, max_new_tokens=max_new_tokens)
                    for record, response_text in zip(records, responses):
                        if "answer" not in record:
                            record["answer"] = {}
                        record["answer"]["value"] = response_text
                        writer.write(pickle.dumps(record))
                        written += 1

                pbar.update(len(records))
                batch_idx += 1
    except KeyboardInterrupt:
        interrupted = True
        print(f"\n⚠ Прервано пользователем (Ctrl+C). Записано {written} из {num_records} записей.")
    finally:
        stop_event.set()
        # Drain очереди, чтобы продюсер не завис на queue.put()
        while not batch_queue.empty():
            try:
                batch_queue.get_nowait()
            except queue.Empty:
                break
        producer.join(timeout=10)
        writer.close()
        reader.close()

    if not interrupted:
        print(f"Готово! Файл сохранен: {output_path} ({written} записей)")

In [ ]:
generate_and_save_batched_dataset(
    model=model,
    params=original_params,
    tokenizer=gm.text.Gemma3Tokenizer(),
    batch_size=2,  # уменьшено с 8 для RTX 2060 SUPER (8 ГБ VRAM)
    split_name=split_name,
    max_new_tokens=max_new_tokens,
    # limit=32 # Ограничим для теста
)

In [ ]:
import array_record.python.array_record_module as array_record
import pickle
import os

# Путь к сгенерированному файлу
generated_path = os.path.join(str(DATA_DIR), 'validation_gemma_generated.array_record')

if os.path.exists(generated_path):
    reader = array_record.ArrayRecordReader(generated_path)
    num_records = reader.num_records()
    print(f"Всего записей в файле: {num_records}")

    # Читаем первые 3 записи для проверки
    for i in range(num_records):
        record_bytes = reader.read()
        record = pickle.loads(record_bytes)
        print(f"\n--- Проверка записи {i} ---")
        print(f"Question: {record.get('question', 'N/A')}")
        print(f"Gemma Answer (saved): {record.get('answer', {}).get('value', 'N/A')}")

    reader.close()
else:
    print(f"Файл не найден: {generated_path}")